In [46]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import pandas as pd
import csv
import os

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# angle calculation
def calculate_angle(point_a, point_b, point_c):
    point_a, point_b, point_c = np.array(point_a), np.array(point_b), np.array(point_c)
    vector_ba = point_a - point_b
    vector_bc = point_c - point_b
    cosine_angle = np.dot(vector_ba, vector_bc) / (np.linalg.norm(vector_ba) * np.linalg.norm(vector_bc))
    return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))

# additional feature preparation
def prepare_ml_features(lm, side, frame_width, frame_height):
    if side == "Right":
        hip, knee, ankle = 24, 26, 28
        shoulder, elbow, wrist = 12, 14, 16
        heel, toe = 30, 32
    else:
        hip, knee, ankle = 23, 25, 27
        shoulder, elbow, wrist = 11, 13, 15
        heel, toe = 29, 31

    # normalization and torso scaling
    mid_hip_x = (lm[24].x + lm[23].x) / 2
    mid_hip_y = (lm[24].y + lm[23].y) / 2
    mid_shoulder_x = (lm[12].x + lm[11].x) / 2
    mid_shoulder_y = (lm[12].y + lm[11].y) / 2

    torso_dist = np.sqrt((mid_shoulder_x - mid_hip_x)**2 + (mid_shoulder_y - mid_hip_y)**2)
    if torso_dist == 0: torso_dist = 1

    def norm_x(l_idx): return (lm[l_idx].x - mid_hip_x) / torso_dist
    def norm_y(l_idx): return (lm[l_idx].y - mid_hip_y) / torso_dist

    # feature calculation
    features = {
        'n_ankle_x': round(norm_x(ankle), 4),
        'n_ankle_y': round(norm_y(ankle), 4),
        'n_knee_x': round(norm_x(knee), 4),
        'n_knee_y': round(norm_y(knee), 4),
        'n_foot_stretch': round(norm_x(ankle) - norm_x(hip), 4),
        'n_heel_toe_slope': round(lm[heel].y - lm[toe].y, 4),
        'n_knee_elevation': round(norm_y(knee), 4),
        'n_shoulder_lean': round(norm_x(shoulder), 4),
        'n_elbow_x': round(norm_x(elbow), 4)
    }
    return features

# video features extraction
def analyze_video(video_path):
    video_capture = cv2.VideoCapture(video_path)
    frames_per_second = video_capture.get(cv2.CAP_PROP_FPS)

    # thresholds and setup
    push_off_threshold = 0.08
    min_swing_frames = frames_per_second * 0.22 
    min_contact_frames = 3
    gct_timeout = 1.0

    all_step_metrics_storage = []
    history_window_size = 10 
    right_knee_angle_history = deque(maxlen=history_window_size)
    left_knee_angle_history = deque(maxlen=history_window_size)
    hip_height_history = deque(maxlen=50)
    cadence_history = deque(maxlen=5)
    gct_filter_history = deque(maxlen=5)

    current_max_split = 0
    leg_is_on_ground = {"Right": False, "Left": False}
    ankle_y_at_contact = {"Right": None, "Left": None}
    last_ankle_y = {"Right": None, "Left": None}

    last_leg_that_landed = None
    last_strike_frame_index = 0
    previous_strike_frame = None
    avg_cadence = 0
    right_step_count = 0
    left_step_count = 0
    current_status_event = ""

    with mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7) as pose_analyzer:
        while video_capture.isOpened():
            ret, frame = video_capture.read()
            if not ret: break

            overlay_layer = frame.copy()
            display_frame = frame.copy()
            frame_height, frame_width, _ = frame.shape
            results = pose_analyzer.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
                def get_pixel_point(idx): return np.array([int(landmarks[idx].x * frame_width), int(landmarks[idx].y * frame_height)])

                # keypoint extraction
                right_hip, right_knee, right_ankle = get_pixel_point(24), get_pixel_point(26), get_pixel_point(28)
                left_hip, left_knee, left_ankle = get_pixel_point(23), get_pixel_point(25), get_pixel_point(27)
                right_shoulder, right_elbow, right_wrist = get_pixel_point(12), get_pixel_point(14), get_pixel_point(16)
                left_shoulder, left_elbow, left_wrist = get_pixel_point(11), get_pixel_point(13), get_pixel_point(15)

                # joint angle calculation
                right_knee_angle = calculate_angle(right_hip, right_knee, right_ankle)
                left_knee_angle = calculate_angle(left_hip, left_knee, left_ankle)
                right_elbow_angle = calculate_angle(right_shoulder, right_elbow, right_wrist)
                left_elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)

                # trunk angle calculation
                trunk_vector = np.array([landmarks[12].x - landmarks[24].x, landmarks[12].y - landmarks[24].y])
                trunk_angle = 180 - np.degrees(np.arctan2(np.abs(trunk_vector[0]), np.abs(trunk_vector[1])))

                # vertical oscillation calculation
                torso_height = np.abs(landmarks[24].y - landmarks[12].y) 
                mid_hip_y = (landmarks[24].y + landmarks[23].y) / 2
                hip_height_history.append(mid_hip_y)
                v_osc = ((max(hip_height_history) - min(hip_height_history)) / torso_height) * 100 if torso_height > 0 else 0

                # leg split calculation
                v_r, v_l = (right_knee - right_hip), (left_knee - left_hip)
                split_angle = np.degrees(np.arccos(np.clip(np.dot(v_r/np.linalg.norm(v_r), v_l/np.linalg.norm(v_l)), -1.0, 1.0)))
                if split_angle > current_max_split: current_max_split = split_angle

                # drawing and overlay
                torso_pts = np.array([right_shoulder, left_shoulder, left_hip, right_hip], np.int32)
                cv2.fillPoly(overlay_layer, [torso_pts], (0, 255, 0)) 
                cv2.addWeighted(overlay_layer, 0.15, display_frame, 0.85, 0, display_frame)
                cv2.polylines(display_frame, [torso_pts], True, (255, 255, 255), 2)
                cv2.line(display_frame, tuple(right_shoulder), tuple(left_hip), (255, 255, 255), 1)
                cv2.line(display_frame, tuple(left_shoulder), tuple(right_hip), (255, 255, 255), 1)
                cv2.line(display_frame, tuple(right_ankle), tuple(left_ankle), (255, 0, 255), 2)

                for h, k, a, c in [(right_hip, right_knee, right_ankle, (0, 255, 0)), (left_hip, left_knee, left_ankle, (0, 255, 255))]:
                    cv2.line(display_frame, tuple(h), tuple(k), c, 3)
                    cv2.line(display_frame, tuple(k), tuple(a), c, 3)
                for s, e, w in [(right_shoulder, right_elbow, right_wrist), (left_shoulder, left_elbow, left_wrist)]:
                    cv2.line(display_frame, tuple(s), tuple(e), (255, 165, 0), 3)
                    cv2.line(display_frame, tuple(e), tuple(w), (255, 165, 0), 3)

                for j, ang, c in [(right_knee, right_knee_angle, (0,255,0)), (left_knee, left_knee_angle, (0,255,255)), (right_elbow, right_elbow_angle, (255,165,0))]:
                    cv2.putText(display_frame, f"{int(ang)}", tuple(j + [10, -10]), 1, 1.2, c, 2)

                right_knee_angle_history.append(right_knee_angle)
                left_knee_angle_history.append(left_knee_angle)
                current_frame = video_capture.get(cv2.CAP_PROP_POS_FRAMES)

                # strike detection logic
                if len(right_knee_angle_history) == history_window_size:
                    middle_index = history_window_size // 2
                    dynamic_lockout = frames_per_second * 0.18 if avg_cadence > 190 else min_swing_frames

                    for side, history, ankle_y, e_ang in [("Right", right_knee_angle_history, landmarks[28].y, right_elbow_angle), ("Left", left_knee_angle_history, landmarks[27].y, left_elbow_angle)]:
                        if (last_leg_that_landed != side and history[middle_index] > 150 and history[middle_index] == max(history) and (current_frame - last_strike_frame_index) > dynamic_lockout):

                            # gct calculation start
                            other_side = "Left" if side == "Right" else "Right"
                            if leg_is_on_ground[other_side]:
                                for rec in reversed(all_step_metrics_storage):
                                    if rec['side'] == other_side and not rec['done']:
                                        raw_val = ((current_frame - rec['start_frame']) / frames_per_second) * 1000
                                        gct_filter_history.append(raw_val)
                                        rec['gct'] = sum(gct_filter_history) / len(gct_filter_history) if len(gct_filter_history) > 0 else raw_val
                                        rec['done'] = True
                                        leg_is_on_ground[other_side] = False
                                        break

                            leg_is_on_ground[side], ankle_y_at_contact[side], last_leg_that_landed = True, ankle_y, side
                            current_status_event = f"{side.upper()} STRIKE"
                            
                            # cadence calculation
                            if previous_strike_frame is not None:
                                cadence_history.append((60 * frames_per_second) / (current_frame - previous_strike_frame))
                                avg_cadence = sum(cadence_history) / len(cadence_history)

                            previous_strike_frame, last_strike_frame_index = current_frame, current_frame
                            if side == "Right": right_step_count += 1
                            else: left_step_count += 1

                            # step storage
                            ml_data = prepare_ml_features(landmarks, side, frame_width, frame_height)
                            all_step_metrics_storage.append({
                                'side': side, 'strike_knee': history[middle_index], 'split': current_max_split, 
                                'trunk': trunk_angle, 'cadence': avg_cadence, 'v_osc': v_osc, 
                                'elbow': e_ang, 'start_frame': current_frame, 'push_knee': history[middle_index], **ml_data, 'done': False, 'gct': 0
                            })
                            current_max_split = 0

                # push knee update and push-off detection
                for side, current_y in [("Right", landmarks[28].y), ("Left", landmarks[27].y)]:
                    if leg_is_on_ground[side]:
                        for rec in reversed(all_step_metrics_storage):
                            if rec['side'] == side and not rec['done']:
                                # push knee update
                                c_ang = right_knee_angle if side == "Right" else left_knee_angle
                                if c_ang > rec['push_knee']: rec['push_knee'] = c_ang

                                frames_on_ground = current_frame - rec['start_frame']
                                if (frames_on_ground / frames_per_second) > gct_timeout:
                                    leg_is_on_ground[side], rec['done'], rec['gct'] = False, True, 0
                                    break

                                # gct calculation end
                                moving_up = (current_y < last_ankle_y[side]) if last_ankle_y[side] is not None else False
                                if (ankle_y_at_contact[side] - current_y) > push_off_threshold and frames_on_ground >= min_contact_frames and moving_up:
                                    raw_val = (frames_on_ground / frames_per_second) * 1000
                                    gct_filter_history.append(raw_val)
                                    rec['gct'] = sum(gct_filter_history) / len(gct_filter_history) if len(gct_filter_history) > 0 else raw_val
                                    rec['done'] = True
                                    leg_is_on_ground[side] = False
                                    current_status_event = f"{side.upper()} PUSH-OFF"
                                    break
                        last_ankle_y[side] = current_y

                # display visualizations
                cv2.rectangle(display_frame, (0,0), (280, 100), (20,20,20), -1)
                cv2.putText(display_frame, f"STEPS: {right_step_count + left_step_count}", (15, 35), 1, 1.8, (255,255,255), 2)
                cv2.putText(display_frame, f"CADENCE: {int(avg_cadence)}", (15, 65), 1, 1.2, (0, 255, 0), 2)
                cv2.putText(display_frame, f"STATUS: {current_status_event}", (15, 95), 1, 1.0, (0, 255, 255), 1)

                for side_ui, pos, state in [("LEFT", (10, frame_height-20), leg_is_on_ground["Left"]), ("RIGHT", (frame_width-130, frame_height-20), leg_is_on_ground["Right"])]:
                    color = (0, 255, 0) if state else (0, 0, 255)
                    cv2.rectangle(display_frame, (pos[0]-10, pos[1]-40), (pos[0]+120, pos[1]+10), (0,0,0), -1)
                    cv2.putText(display_frame, side_ui, (pos[0], pos[1]-20), 1, 1.2, color, 2)
                    cv2.putText(display_frame, "CONTACT" if state else "FLIGHT", (pos[0], pos[1]), 1, 0.9, (255,255,255), 1)

                cv2.imshow('Running analysis', display_frame)
                if cv2.waitKey(1) & 0xFF == ord('q'): break

    video_capture.release()
    cv2.destroyAllWindows()
    return all_step_metrics_storage

In [47]:
# baseline metric score intervals
def get_metric_score(metric, val):
    
    if metric == "gct":
        if val < 270: return 3         
        if 270 <= val <= 285: return 2
        if 285 <= val <= 300: return 1
        return 0
    
    if metric == "cadence":
        if val > 180: return 3         
        if 170 <= val <= 180: return 2
        if 160 <= val <= 170: return 1
        return 0
    
    if metric == "push_knee":
        if val > 165: return 3         
        if 155 <= val <= 165: return 2
        if 150 <= val <= 155: return 1
        return 0
    
    if metric == "strike_knee":
        if 155 <= val <= 165: return 3
        if 150 <= val <= 170: return 2
        if 145 <= val <= 175: return 1
        return 0
    
    if metric == "elbow":
        if 70 <= val <= 100: return 3
        if 60 <= val <= 115: return 2
        if 50 <= val <= 120: return 1
        return 0
    
    if metric == "v_osc":
        if val < 15.0: return 3
        if val < 18.0: return 2
        if val < 21.0: return 1
        return 0

    if metric == "trunk":
        if 160 <= val <= 180: return 3
        if 150 <= val <= 185: return 2
        if 145 <= val <= 190: return 1
        return 0

    if metric == "split":
        if 65 <= val <= 75: return 3
        if 60 <= val <= 80: return 2
        if 55 <= val <= 85: return 1
        return 0
    return 0

In [48]:
# csv storing
fieldnames = [
    'rep_number', 'frame_index', 'file_name', 'side',
    'cadence_value (spm)', 'gct_value (ms)', 'vert_osc_value (%)', 
    'knee_strike_angle (deg)', 'knee_push_angle (deg)', 'elbow_angle_val (deg)', 
    'leg_split_val (deg)', 'trunk_lean_value (deg)',
    'n_ankle_x', 'n_ankle_y', 'n_knee_x', 'n_knee_y', 
    'n_foot_stretch', 'n_heel_toe_slope', 'n_knee_elevation', 
    'n_shoulder_lean', 'n_elbow_x',
    'cadence_score', 'gct_score', 'vert_osc_score', 'knee_strike_score', 
    'knee_push_score', 'elbow_angle_score', 'leg_split_score', 'trunk_lean_score'
]

def save_steps_to_csv(steps_list, video_path, output_file='side_view_dataset.csv'):
    # file setup
    file_exists = os.path.isfile(output_file)
    
    with open(output_file, 'a', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()

        line_count = 0
        if file_exists:
            with open(output_file, 'r') as f:
                line_count = sum(1 for _ in f) - 1

        for i, m in enumerate(steps_list, start=max(0, line_count) + 1):
            if not m.get('done'): continue
            
            # row data mapping
            row = {
                'rep_number': i,
                'frame_index': int(m['start_frame']),
                'file_name': video_path,
                'side': m['side'],
                'cadence_value (spm)': round(m['cadence'], 2),
                'gct_value (ms)': int(m['gct']),
                'vert_osc_value (%)': round(m['v_osc'], 2),
                'knee_strike_angle (deg)': int(m['strike_knee']),
                'knee_push_angle (deg)': int(m['push_knee']),
                'elbow_angle_val (deg)': int(m['elbow']),
                'leg_split_val (deg)': int(m['split']),
                'trunk_lean_value (deg)': int(m['trunk']),
                
                # ml feature mapping
                'n_ankle_x': m.get('n_ankle_x'),
                'n_ankle_y': m.get('n_ankle_y'),
                'n_knee_x': m.get('n_knee_x'),
                'n_knee_y': m.get('n_knee_y'),
                'n_foot_stretch': m.get('n_foot_stretch'),
                'n_heel_toe_slope': m.get('n_heel_toe_slope'),
                'n_knee_elevation': m.get('n_knee_elevation'),
                'n_shoulder_lean': m.get('n_shoulder_lean'),
                'n_elbow_x': m.get('n_elbow_x'),
                
                # score calculation - labels
                'cadence_score': get_metric_score("cadence", m['cadence']),
                'gct_score': get_metric_score("gct", m['gct']),
                'vert_osc_score': get_metric_score("v_osc", m['v_osc']),
                'knee_strike_score': get_metric_score("strike_knee", m['strike_knee']),
                'knee_push_score': get_metric_score("push_knee", m['push_knee']),
                'elbow_angle_score': get_metric_score("elbow", m['elbow']),
                'leg_split_score': get_metric_score("split", m['split']),
                'trunk_lean_score': get_metric_score("trunk", m['trunk'])
            }
            writer.writerow(row)
            
    print(f"Uspješno spremljeno {len(steps_list)} koraka.")

In [49]:
# dataset creation
paths = ['./Videos_side_view/side_view2.mov']

for video_path in paths:
    # data processing
    captured_data = analyze_video(video_path)

    # dataset saving and scoring
    if captured_data:
        save_steps_to_csv(captured_data, video_path)

print("Dataset ažuriran.")

I0000 00:00:1769216622.545852 3913461 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
W0000 00:00:1769216622.615344 4031957 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769216622.626055 4031956 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Uspješno spremljeno 327 koraka.
--- proces završen. dataset je ažuriran! ---


In [53]:
# model training and evaluating
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# loading data
df = pd.read_csv('side_view_dataset.csv')

# feature and target definition
X_features = [
    'cadence_value (spm)', 'gct_value (ms)', 'vert_osc_value (%)', 
    'knee_strike_angle (deg)', 'knee_push_angle (deg)', 'elbow_angle_val (deg)', 
    'leg_split_val (deg)', 'trunk_lean_value (deg)',
    'n_ankle_x', 'n_ankle_y', 'n_knee_x', 'n_knee_y', 
    'n_foot_stretch', 'n_heel_toe_slope', 'n_knee_elevation', 
    'n_shoulder_lean', 'n_elbow_x'
]

target_scores = [
    'cadence_score', 'gct_score', 'vert_osc_score', 'knee_strike_score', 
    'knee_push_score', 'elbow_angle_score', 'leg_split_score', 'trunk_lean_score'
]

# data cleaning
df_clean = df[df['cadence_value (spm)'] > 0].dropna()
X = df_clean[X_features]
Y = df_clean[target_scores]

# train test split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# model training and evaluation
print(f"═" * 40)
print(f"{'METRIKA':<20} | {'TOČNOST':<10}")
print(f"─" * 40)

models = {}
for target in target_scores:
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, Y_train[target])
    
    y_pred = clf.predict(X_test)
    acc = accuracy_score(Y_test[target], y_pred)
    
    models[target] = clf
    print(f"{target:<20} | {acc:.2%}")

print(f"═" * 40)

# sample prediction
sample_idx = 0
real_values = Y_test.iloc[sample_idx].values
predicted_values = [models[t].predict(X_test.iloc[[sample_idx]])[0] for t in target_scores]

print("\nprovjera na jednom koraku:")
print(f"stvarne ocjene:    {real_values}")
print(f"predviđene ocjene: {predicted_values}")

════════════════════════════════════════
METRIKA              | TOČNOST   
────────────────────────────────────────
cadence_score        | 98.48%
gct_score            | 98.48%
vert_osc_score       | 96.97%
knee_strike_score    | 100.00%
knee_push_score      | 100.00%
elbow_angle_score    | 90.91%
leg_split_score      | 96.97%
trunk_lean_score     | 100.00%
════════════════════════════════════════

provjera na jednom koraku:
stvarne ocjene:    [0 0 0 1 3 2 0 3]
predviđene ocjene: [0, 0, 0, 1, 3, 3, 0, 3]


In [54]:
# model testing
def test_on_new_video(video_path, trained_models, features_list):
    print(f"--- analiza videa: {video_path} ---")
    
    # video analysis and data extraction
    raw_steps_data = analyze_video(video_path)
    if not raw_steps_data:
        print("nema detektiranih koraka.")
        return

    # dataframe preparation
    test_df = pd.DataFrame(raw_steps_data)
    test_df = test_df[test_df['done'] == True].copy()

    # column mapping for model compatibility
    column_mapping_for_model = {
        'cadence': 'cadence_value (spm)',
        'gct': 'gct_value (ms)',
        'v_osc': 'vert_osc_value (%)',
        'strike_knee': 'knee_strike_angle (deg)',
        'push_knee': 'knee_push_angle (deg)',
        'elbow': 'elbow_angle_val (deg)',
        'split': 'leg_split_val (deg)',
        'trunk': 'trunk_lean_value (deg)'
    }
    model_df = test_df.rename(columns=column_mapping_for_model)

    # target mapping for math score comparison
    target_to_math_key = {
        'cadence_score': 'cadence',
        'gct_score': 'gct',
        'vert_osc_score': 'v_osc',
        'knee_strike_score': 'strike_knee',
        'knee_push_score': 'push_knee',
        'elbow_angle_score': 'elbow',
        'leg_split_score': 'split',
        'trunk_lean_score': 'trunk'
    }

    print(f"\n{'metrika':<20} | {'predviđanje':<12} | {'matematički izračun'}")
    print("-" * 55)
    
    # evaluation loop
    for target, model in trained_models.items():
        # model prediction
        X_input = model_df[features_list]
        ai_preds = model.predict(X_input)
        avg_ai_score = int(round(np.mean(ai_preds)))
        
        # math score comparison
        math_key = target_to_math_key.get(target)
        if math_key in test_df.columns:
            avg_val = test_df[math_key].mean()
            your_score = get_metric_score(math_key, avg_val)
        else:
            your_score = "N/A"
            
        print(f"{target:<20} | {avg_ai_score:<12} | {your_score}")

# pokretanje
novi_video = './Videos_side_view/Video.mov'
test_on_new_video(novi_video, models, X_features)

--- analiza videa: ./Videos_side_view/Video.mov ---


I0000 00:00:1769217072.604165 3913461 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
W0000 00:00:1769217072.656946 4042146 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769217072.668385 4042146 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.



metrika              | predviđanje  | matematički izračun
-------------------------------------------------------
cadence_score        | 1            | 2
gct_score            | 2            | 3
vert_osc_score       | 1            | 1
knee_strike_score    | 3            | 3
knee_push_score      | 2            | 2
elbow_angle_score    | 2            | 2
leg_split_score      | 2            | 3
trunk_lean_score     | 3            | 3
